# SA BankAccount — Phase 2 final submission

Same Kaggle CPU session, PostgreSQL and Node.js for both builds.
Five scenarios, three 60-second measured runs per variant, with 2-second warm-up.

- Original: `c347c921d77ba533cfacd8e180feccef720be013`
- Final: `bf78c1e17f4b89e661ea5c28dfa99f60afab47f4`
- Compact transfer writes on, API admission queue off.
- History reads the same 20 IDs using OFFSET + COUNT (original) and a known cursor (final).
- The output contains raw runs, API CPU/RSS, reconciliation checks and the test log.


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq git ca-certificates postgresql-16 postgresql-client-16 nodejs npm
mkdir -p /tmp/bank-phase2-tools
npm install --prefix /tmp/bank-phase2-tools/node-runtime --no-audit --no-fund node@24.22.3
/tmp/bank-phase2-tools/node-runtime/node_modules/node/bin/node --version


In [ ]:
import os, json, subprocess, tempfile, time, shutil, socket
from pathlib import Path

os.environ['PATH']='/tmp/bank-phase2-tools/node-runtime/node_modules/node/bin:'+os.environ['PATH']
BASELINE='c347c921d77ba533cfacd8e180feccef720be013'
FINAL='bf78c1e17f4b89e661ea5c28dfa99f60afab47f4'
ROOT=Path(tempfile.mkdtemp(prefix='sa-bank-phase2-final-',dir='/tmp'))
OUT=Path('/kaggle/working/sa-bankaccount-final-results')
(OUT/'results').mkdir(parents=True,exist_ok=True)

def run(args, cwd=ROOT):
    subprocess.run(args,cwd=cwd,check=True)

run(['git','clone','https://github.com/FlowBoat123/SA_BankAccount.git','.'])
run(['git','checkout','--detach',FINAL])
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()==FINAL
pgbin='/usr/lib/postgresql/16/bin/'
pgdata=tempfile.mkdtemp(prefix='bank-phase2-final-pg-')
run(['chown','postgres:postgres',pgdata])
run(['runuser','-u','postgres','--',pgbin+'initdb','-D',pgdata,'-A','trust','--encoding=UTF8','--locale=C'])
with socket.socket() as probe:
    probe.bind(('127.0.0.1',55432))
run(['runuser','-u','postgres','--',pgbin+'pg_ctl','-D',pgdata,'-l',pgdata+'/server.log','-o','-h 127.0.0.1 -p 55432','-w','start'])
run([pgbin+'createdb','-h','127.0.0.1','-p','55432','-U','postgres','bank_phase2_final_test'])
os.environ.update(
    DATABASE_URL='postgresql://postgres@127.0.0.1:55432/bank_phase2_final_test',
    OUTPUT=str(OUT/'results'), ROUNDS='3', DURATION='60',
    HOT_CLIENTS='10,50,100',
    SCENARIOS='transfer_hot_10,transfer_hot_50,transfer_hot_100,transfer_distributed_10,history_300000_deep',
    HISTORY_SIZE='300000', MEASUREMENT_ENVIRONMENT='Kaggle CPU',
    BENCHMARK_FINAL_COMMIT=FINAL, TRANSFER_ADMISSION='off', TRANSFER_COMPACT_WRITE='on',
    UV_THREADPOOL_SIZE='4')

limits={}
for key in ['cpu.max','cpuset.cpus.effective','memory.max']:
    f=Path('/sys/fs/cgroup')/key
    limits[key]=f.read_text().strip() if f.exists() else 'unavailable'
limits['lscpu']=subprocess.check_output(['lscpu'],text=True)
limits['postgresSettings']=subprocess.check_output([pgbin+'psql',os.environ['DATABASE_URL'],'-Atc',
    "SELECT name||'='||setting FROM pg_settings WHERE name IN ('max_connections','shared_buffers','work_mem','fsync','synchronous_commit') ORDER BY name"],text=True)
(OUT/'results'/'kaggle-limits.json').write_text(json.dumps(limits,indent=2))

run(['npm','ci'])
run(['node','node_modules/prisma/build/index.js','generate'])
run(['node','node_modules/prisma/build/index.js','migrate','deploy'])
run(['npm','run','phase2:baseline'])
run(['node','scripts/phase2/verify.cjs'])


In [ ]:
from statistics import median

try:
    run(['npm','run','phase2:benchmark'])
    data=json.loads((OUT/'results'/'runs.json').read_text())
    assert len(data)==30 and len({(r['round'],r['scenario'],r['variant']) for r in data})==30
    writes=[r for r in data if r['scenario'].startswith('transfer_')]
    assert len(writes)==24 and all(r['reconciliation']['conserved'] and r['reconciliation']['ledgerMatchesBalances'] for r in writes)
    results=[]
    for scenario in dict.fromkeys(r['scenario'] for r in data):
        row={'scenario':scenario}
        for variant in ['baseline','improved']:
            group=[r for r in data if r['scenario']==scenario and r['variant']==variant]
            row[variant]={
                'rounds':len(group),
                **{k:median(r[k] for r in group) for k in ['p50','p95','p99']},
                'p95Min':min(r['p95'] for r in group), 'p95Max':max(r['p95'] for r in group),
                'rps':median(r['successRps'] for r in group),
                'cpuPercent':median(r['apiCpuPercent'] for r in group),
                'rssMiB':median(r['apiRssBytes'] for r in group)/1024**2,
                'requests':sum(r['requests'] for r in group),
                'errors':sum(r['requests']-r['successes'] for r in group),
                'committedWithoutAck':sum(r.get('reconciliation',{}).get('committedWithoutAck',0) for r in group),
            }
        results.append(row)
        print(scenario, json.dumps(row,ensure_ascii=False))
    (OUT/'results'/'summary.json').write_text(json.dumps(results,indent=2)+'\n')
    (OUT/'COMPLETE.json').write_text(json.dumps({'baseline':BASELINE,'final':FINAL,'runs':len(data),'writeRunsReconciled':len(writes),'requests':sum(r['requests'] for r in data),'errors':sum(r['requests']-r['successes'] for r in data)},indent=2)+'\n')
finally:
    archive=shutil.make_archive('/kaggle/working/sa-bankaccount-phase2-final','zip',OUT)
    print('Kaggle output:',archive)
    run(['runuser','-u','postgres','--',pgbin+'pg_ctl','-D',pgdata,'-m','fast','-w','stop'])
